### Ноутбук "EDA и подготовка данных"

#### Описание

Генерация и загрузка синтетических данных в PostgreSQL, первичная проверка целостности перед основным анализом.

##### Импорт модулей и библиотек

In [1]:
import sys
sys.path.append("../src")

from generate_data import generate_users, generate_subscription, generate_payments, generate_ab_assignments, generate_events

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import text

from db_connection import get_engine
engine = get_engine()

##### Создание таблицы "users"

Генерация датафрейма "users_df" с 3000 пользователей, без столбца "id"

In [3]:
users_df = generate_users(3000)
display(users_df.head())
display(users_df.info())

,acquisition_channel,plan,country,signup_date
0,social,free,China,2025-05-05
1,organic,free,Spain,2025-10-21
2,referral,pro,Russia,2026-01-22
3,organic,free,Russia,2025-04-26
4,organic,free,Russia,2026-03-08


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   acquisition_channel  3000 non-null   str   
 1   plan                 3000 non-null   str   
 2   country              3000 non-null   str   
 3   signup_date          3000 non-null   object
dtypes: object(1), str(3)
memory usage: 93.9+ KB


None

*Очистка таблицы "users"*

In [4]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE users RESTART IDENTITY CASCADE;"))

Генерация датафрейма "users_db" со столбцом "id"

In [5]:
users_df.to_sql("users", engine, if_exists="append", index=False)

users_db = pd.read_sql("SELECT * FROM users;", engine)
display(users_db.columns)
display(users_db.shape)

Index(['id', 'acquisition_channel', 'plan', 'country', 'signup_date'], dtype='str')

(3000, 5)

##### Создание таблицы "subscription"

Генерация датафрейма "subscriptions_df" без столбца "id"

In [6]:
subscriptions_df = generate_subscription(users_db)
display(subscriptions_df.head())
display(subscriptions_df.info())

,user_id,plan,price,start_date,end_date,status
0,3,pro,250.0,2026-01-24,NaT,active
1,7,basic,100.0,2025-07-27,NaT,active
2,8,basic,100.0,2025-06-04,NaT,active
3,11,pro,250.0,2026-01-18,NaT,active
4,16,basic,100.0,2026-04-09,2026-11-11,canceled


<class 'pandas.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     1328 non-null   int64         
 1   plan        1328 non-null   str           
 2   price       1328 non-null   float64       
 3   start_date  1328 non-null   datetime64[us]
 4   end_date    432 non-null    datetime64[us]
 5   status      1328 non-null   str           
dtypes: datetime64[us](2), float64(1), int64(1), str(2)
memory usage: 62.4 KB


None

*Очистка таблицы "subscriptions"*

In [7]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE subscriptions RESTART IDENTITY CASCADE;"))

Генерация датафрейма "subscriptions_db" со столбцом "id"

In [8]:
subscriptions_df.to_sql("subscriptions", engine, if_exists="append", index=False)

subscriptions_db = pd.read_sql("SELECT * FROM subscriptions;", engine)
display(subscriptions_db.columns)
display(subscriptions_db.shape)

Index(['id', 'user_id', 'plan', 'price', 'start_date', 'end_date', 'status'], dtype='str')

(1328, 7)

##### Создание таблицы "payments"

Генерация датафрейма "payments_df" без столбца "id"

In [9]:
payments_df = generate_payments(subscriptions_db)
display(payments_df.head())
display(payments_df.info())

,subscription_id,amount,payment_date,status
0,1,250.0,2026-01-24,succeeded
1,1,250.0,2026-02-23,succeeded
2,1,250.0,2026-03-25,succeeded
3,1,250.0,2026-04-24,succeeded
4,1,250.0,2026-05-24,succeeded


<class 'pandas.DataFrame'>
RangeIndex: 10837 entries, 0 to 10836
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  10837 non-null  int64         
 1   amount           10837 non-null  float64       
 2   payment_date     10837 non-null  datetime64[us]
 3   status           10837 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 338.8 KB


None

*Очистка таблицы "payments"*

In [10]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE payments RESTART IDENTITY CASCADE;"))

Генерация датафрейма "payments_db" со столбцом "id"

In [11]:
payments_df.to_sql("payments", engine, if_exists="append", index=False)

payments_db = pd.read_sql("SELECT * FROM payments;", engine)
display(payments_db.columns)
display(payments_db.shape)

Index(['id', 'subscription_id', 'amount', 'payment_date', 'status'], dtype='str')

(10837, 5)

##### Создание таблицы "ab_test_assignments"

Генерация датафрейма "assignments_df" без столбца "id"

In [12]:
assignments_df = generate_ab_assignments(users_db)
display(assignments_df.head())
display(assignments_df.info())

,user_id,test,variant,assigned_at
0,9,onboarding,A,2026-09-11
1,9,pricing,B,2026-09-13
2,18,onboarding,A,2026-09-09
3,18,pricing,B,2026-09-11
4,25,onboarding,B,2026-08-23


<class 'pandas.DataFrame'>
RangeIndex: 1060 entries, 0 to 1059
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      1060 non-null   int64         
 1   test         1060 non-null   str           
 2   variant      1060 non-null   str           
 3   assigned_at  1060 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 33.3 KB


None

*Очистка таблицы "ab_test_assignments"*

In [13]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE ab_test_assignments RESTART IDENTITY CASCADE;"))

Генерация датафрейма "assignments_db" со столбцом "id"

In [14]:
assignments_df.to_sql("ab_test_assignments", engine, if_exists="append", index=False)

assignments_db = pd.read_sql("SELECT * FROM ab_test_assignments;", engine)
display(assignments_db.columns)
display(assignments_db.shape)

Index(['id', 'user_id', 'test', 'variant', 'assigned_at'], dtype='str')

(1060, 5)

In [15]:
for table in ['users', 'subscriptions', 'payments', 'events', 'ab_test_assignments']:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {table};", engine)
    print(table, count.iloc[0,0])

users 3000
subscriptions 1328
payments 10837
events 0
ab_test_assignments 1060


##### Создание таблицы "events"

Генерация датафрейма "events_df" без столбца "id"

In [16]:
events_df = generate_events(users_db)
display(events_df.head())
display(events_df.info())

,user_id,event_type,event_time
0,1,signup,2025-05-05 00:00:00
1,2,signup,2025-10-21 00:00:00
2,3,signup,2026-01-22 00:00:00
3,3,onboarding_done,2026-01-22 02:00:00
4,4,signup,2025-04-26 00:00:00


<class 'pandas.DataFrame'>
RangeIndex: 6700 entries, 0 to 6699
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     6700 non-null   int64         
 1   event_type  6700 non-null   str           
 2   event_time  6700 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 157.2 KB


None

*Очистка таблицы "events"*

In [17]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE events RESTART IDENTITY CASCADE;"))

Генерация датафрейма "events_db" со столбцом "id"

In [18]:
events_df.to_sql("events", engine, if_exists="append", index=False)

events_db = pd.read_sql("SELECT * FROM events", engine)
display(events_db.columns)
display(events_db.shape)

Index(['id', 'user_id', 'event_type', 'event_time'], dtype='str')

(6700, 4)